In [1]:
import pandas as pd
import os

In [2]:
RAW_PATH = os.path.join("data", 'raw')
file_path = os.path.join(RAW_PATH, "market_prices.csv")

In [5]:
prices = pd.read_csv(file_path, index_col=0, parse_dates=True)

In [6]:
# Daily Returns
returns = prices.pct_change()

In [8]:
returns.columns = [
    f"{col}_ret_1d"
    for col in returns.columns
]

In [10]:
returns.head()

,GLD_ret_1d,SPY_ret_1d,TLT_ret_1d,VIX_ret_1d
Date,,,,
2005-01-03,NaN,NaN,NaN,NaN
2005-01-04,-0.006509,-0.012220,-0.010480,-0.007102
2005-01-05,-0.001638,-0.006901,0.005353,0.007868
2005-01-06,-0.012186,0.005085,0.000680,-0.036196
2005-01-07,-0.007355,-0.001433,0.002264,-0.006627


In [11]:
# Annualized Rolling Volatility

spy_ret = prices["SPY"].pct_change()

vol_20 = (
    spy_ret
    .rolling(20)
    .std()
    * (252 ** 0.5)
)

In [12]:
features = pd.DataFrame(index=prices.index)

features["SPY_vol_20"] = vol_20

In [14]:
# 50 Day Moving Average

ma50 = prices["SPY"].rolling(50).mean()

features["SPY_price_ma50"] = (
    prices["SPY"] / ma50
)

In [ ]:
# 200 Day MA

ma200 = prices["SPY"].rolling(200).mean()

features["SPY_price_ma200"] = (
    prices["SPY"] / ma200
)   

In [16]:
# Drawdown

rolling_max = prices["SPY"].cummax()

drawdown = (
    prices["SPY"]
    /
    rolling_max
) - 1

In [17]:
features["SPY_drawdown"] = drawdown

In [18]:
features["VIX_level"] = prices["VIX"]

In [19]:
features["TLT_ret_20"] = (
    prices["TLT"]
    .pct_change(20)
)

features["GLD_ret_20"] = (
    prices["GLD"]
    .pct_change(20)
)

In [20]:
features["SPY_ret_5"] = (
    prices["SPY"]
    .pct_change(5)
)

features["SPY_ret_20"] = (
    prices["SPY"]
    .pct_change(20)
)

In [21]:
features = features.dropna()

In [22]:
features.head()

,SPY_vol_20,SPY_price_ma50,SPY_price_ma200,SPY_drawdown,VIX_level,TLT_ret_20,GLD_ret_20,SPY_ret_5,SPY_ret_20
Date,,,,,,,,,
2005-10-17,0.102647,0.979883,0.999383,-0.040915,14.670000,-0.018791,0.021838,0.004301,-0.032334
2005-10-18,0.105068,0.969958,0.988602,-0.051302,15.330000,-0.020250,0.016656,-0.005150,-0.034658
2005-10-19,0.119602,0.986595,1.004947,-0.035520,13.500000,-0.028844,-0.015300,0.019404,-0.009346
2005-10-20,0.133068,0.970037,0.987199,-0.052510,16.110001,-0.025324,-0.008197,0.002044,-0.030245
2005-10-21,0.134228,0.974659,0.991018,-0.048807,16.129999,-0.008850,0.005186,-0.004551,-0.027256


In [25]:
PROCESSED_PATH = os.path.join("data", 'processed')
processed_file_path = os.path.join(PROCESSED_PATH, "features.csv")
os.makedirs(PROCESSED_PATH, exist_ok=True)
features.to_csv(processed_file_path)